# Creating Cherry Rainbow Tables

For N = 2**16, currently only making one table

## Imports

In [2]:
import pickle
import random
from hashlib import sha256
import mmh3
from tqdm import tqdm
from math import pi, sqrt, e, log

## Table Parameters

### Initialise Startpoints

In [ ]:
# initialise startpoints - either generate them or load from pickle - to keep same across runs
def get_startpoints(N, m_0, nlabel, alpha):
    # try opening pickle file, else generate and save
    try:
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'rb') as f:
            startpoints = pickle.load(f)

    # no file found - generate and save
    except FileNotFoundError:
        # random but unique - store as a set?
        startpoints = set()
        while len(startpoints) < m_0:
            startpoints.add(random.randint(0, N-1))

        # store startpoints in pickle file
        with open(f'startpoints_N_{nlabel}_alpha_{alpha}.pkl', 'wb') as f:
            pickle.dump(startpoints, f)

    # return the startpoints
    return startpoints

### Parameters

In [ ]:
# label N to find easier - label is the exponent
nlabel = 16
##############################################################################
N = 2 ** 16 # keyspace
p = 1 - e ** -2 # our table coverage - 86%
##############################################################################
t = round(log(1-p)/log(1-N**(-1/3))) # chain length t
alpha = 0.95 # maximality factor
mt_target = N**(2/3) # our target mt
m_0 = round(mt_target/(1-alpha))    # m_0 - number of startpoints
##############################################################################
# initialise startpoints 
startpoints = get_startpoints(N, m_0, nlabel, alpha)


### Cherry-picks per column - Kis

In [ ]:
# get # cherry-picks per column from pickle file 

# load pickle file
with open(f'higher_costs_ftol_1.pickle', 'rb') as f:
    data = pickle.load(f)

# structure of file
# array of different alpha used
    # for each alpha, array of different costs used
    # for each cost, array arranged as [Kjs, m_0, final_cost, m_values]

# to get Kjs for alpha = 0.95 and cost factor = 25
Kis = data[-1][4][0]

[1.00000000e+00 4.05638693e+03 4.84198287e+03 5.53893372e+03
 6.17010258e+03 6.76131662e+03 7.30317568e+03 7.81400189e+03
 8.29209973e+03 8.74613043e+03 9.17354125e+03 9.59090306e+03
 9.97955904e+03 1.03542539e+04 1.07137863e+04 1.10615641e+04
 1.13932850e+04 1.17152210e+04 1.20221925e+04 1.23221981e+04
 1.26108540e+04 1.28980161e+04 1.31724518e+04 1.34357398e+04
 1.36921129e+04 1.39420270e+04 1.41897938e+04 1.44257535e+04
 1.46599852e+04 1.48858118e+04 1.51129205e+04 1.53274788e+04
 1.55420905e+04 1.57500358e+04 1.59537345e+04 1.61563568e+04
 1.63510499e+04 1.65415571e+04 1.67311870e+04 1.69163693e+04
 1.70981983e+04 1.72748034e+04 1.74539631e+04 1.76268798e+04
 1.77955104e+04 1.79659735e+04 1.81306345e+04 1.82969966e+04
 1.84540559e+04 1.86101691e+04 1.87665396e+04 1.89216704e+04
 1.90721307e+04 1.92238312e+04 1.93700312e+04 1.95185891e+04
 1.96617368e+04 1.98041822e+04 1.99461756e+04 2.00866913e+04
 2.02260216e+04 2.03595422e+04 2.04957203e+04 2.06250707e+04
 2.07600479e+04 2.089303

## Hash and Reduction Functions

In [ ]:
# Hash function
def H(x):
	return int(sha256(bytes(x)).hexdigest(), 16)

# Reduction function
# currently mod but should change to murmurhash in future
def r(y, i, ell=0):   # also takes in ell - number of tables - for future use (but currently ell=0)
	return (y + i + ell*t) % N

## Building the Table

In [ ]:
# take in m_0 and t as parameters - how many chains to start with and how long to make the chains
# store the table as a dictionary of endpoint:startpoint pairs (rather than sp:ep for easier lookup later)
# store in a pickle file
# need to also store the reduction function used in each column - how much more memory is taken up? only need to store index per column, so t more bits of memory

def build_cherry_table(startpoints, Kis, t, N, nlabel, alpha):
    # store table in dictionary
    table = {}  # store all the points then remove duplicate entries - can't do duplicate keys in dictionary anyway so we can just store all ep:sp

    # Instead of making the table chain by chain, we have to make it column by column to test what reduction function to choose
    # TODO: store reduction function indexes

    # hash all current points
    hashed_points = [H(sp) for sp in startpoints]


    # for each column in the table
    for i in range(t):
        
        """
        - for each column
            - hash the current points
            - get the # cherry-picks for that column
            - use each reduction function on the points, and store the best running m_i+1 and reduction function index
                - is it quicker to store the reduced points and overwrite them each time, or tally the points as we test and then reduce all points at the end with best r - but then we're doing m_i more reductions?
            - for each rf sample:
                - reduce the column and store in a set
                - get the length of the set - if its better than before then note down this rf 
                - clear the set and do it again
            - with our best rf, 
        """

        # get # cherry-picks for this column
        k_i = Kis[i]

        # variables to store best reduction function info
        best_m_i_plus_1 = -1
        best_rf_index = -1

        # set to store reduced points for each rf test
        reduced_points_set = set()

        # test each reduction function
        for rf_index in range(k_i):
            # reduce all hashed points with this reduction function
            for hp in hashed_points:
                rp = r(hp, rf_index)  # reduce point
                reduced_points_set.add(rp)   # add to set

            # get m_i+1
            m_i_plus_1 = len(reduced_points_set)

            # check if best
            if m_i_plus_1 > best_m_i_plus_1:
                best_m_i_plus_1 = m_i_plus_1
                best_rf_index = rf_index

            # clear set for next rf test
            reduced_points_set.clear()

        # with best rf, reduce all hashed points and update for next column
        reduced = []
        for hp in hashed_points:
            rp = r(hp, best_rf_index)  # reduce point
            reduced.add(rp)   # add to set

        # store current points and startpoints for next column
        


    return table